# Donut visual-token analysis

This notebook studies the final Swin tokens of one trained Donut model: their distributions, spatial structure, redundancy, similarity, correlation, and the decoder cross-attention they receive.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

from donut import DonutModel
from donut.model import decoder_start_ids

CHECKPOINT = Path("/path/to/trained/checkpoint")
IMAGE_PATH = Path("/path/to/image.png")
MAX_NEW_TOKENS = 32
MAX_SIMILARITY_TOKENS = 512
SEED = 42

## Produce the visual tokens

The eager backend is intentional: unlike fused SDPA, it exposes decoder attention matrices.

In [ ]:
donut = DonutModel.load(str(CHECKPOINT), attention_backend="eager")
model = donut.model
device = next(model.parameters()).device
dtype = next(model.parameters()).dtype

image = Image.open(IMAGE_PATH).convert("RGB")
pixels = donut.processor(image, return_tensors="pt").pixel_values.to(
    device=device, dtype=dtype
)

with torch.inference_mode():
    encoder_outputs = model.encoder(pixels, return_dict=True)

tokens = encoder_outputs.last_hidden_state[0].float().cpu()
grid_height = (pixels.shape[-2] + 31) // 32
grid_width = (pixels.shape[-1] + 31) // 32
token_grid = tokens.reshape(grid_height, grid_width, -1)
tokens.shape, token_grid.shape

## Basic statistical properties

In [ ]:
token_norms = tokens.norm(dim=1)
feature_stds = tokens.std(dim=0)

pd.Series({
    "number_of_tokens": tokens.shape[0],
    "embedding_dimension": tokens.shape[1],
    "global_mean": tokens.mean().item(),
    "global_std": tokens.std().item(),
    "mean_token_norm": token_norms.mean().item(),
    "std_token_norm": token_norms.std().item(),
    "mean_feature_std": feature_stds.mean().item(),
    "near_zero_fraction": (tokens.abs() < 1e-3).float().mean().item(),
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(tokens.flatten().numpy(), bins=100)
axes[0].set_title("All token values")
axes[1].hist(token_norms.numpy(), bins=50)
axes[1].set_title("Token norms")
axes[2].hist(feature_stds.numpy(), bins=50)
axes[2].set_title("Standard deviation by feature")
plt.tight_layout()

## Spatial structure and PCA

PCA indicates whether the token matrix can be represented in fewer feature dimensions. Spatial maps show where token magnitude and the leading variation occur.

In [ ]:
centered = tokens - tokens.mean(dim=0)
_, singular_values, directions = torch.pca_lowrank(centered, q=32)
scores = centered @ directions
explained = singular_values.square() / centered.square().sum()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].imshow(token_norms.reshape(grid_height, grid_width), cmap="viridis")
axes[0].set_title("Token norm")
axes[1].imshow(scores[:, 0].reshape(grid_height, grid_width), cmap="coolwarm")
axes[1].set_title("First principal component")
axes[2].plot(explained.cumsum(0).numpy(), marker="o")
axes[2].set(title="Cumulative explained variance", xlabel="Components", ylabel="Fraction")
plt.tight_layout()

## Similarity and correlation

Cosine similarity compares token direction. Pearson correlation compares their centered feature patterns. For large images, a reproducible subset is used to keep the matrices manageable.

In [ ]:
generator = torch.Generator().manual_seed(SEED)
count = min(MAX_SIMILARITY_TOKENS, len(tokens))
indices = torch.randperm(len(tokens), generator=generator)[:count]
sample = tokens[indices]

cosine_similarity = F.normalize(sample, dim=1) @ F.normalize(sample, dim=1).T
pearson_correlation = torch.corrcoef(sample)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cosine_similarity, vmin=-1, vmax=1, cmap="coolwarm")
axes[0].set_title("Token cosine similarity")
axes[1].imshow(pearson_correlation, vmin=-1, vmax=1, cmap="coolwarm")
axes[1].set_title("Token Pearson correlation")
plt.tight_layout()

In [ ]:
right_similarity = F.cosine_similarity(token_grid[:, :-1], token_grid[:, 1:], dim=-1)
down_similarity = F.cosine_similarity(token_grid[:-1], token_grid[1:], dim=-1)
off_diagonal = ~torch.eye(count, dtype=torch.bool)
nearest_similarity = cosine_similarity.masked_fill(~off_diagonal, -1).max(dim=1).values

pd.Series({
    "mean_random_pair_cosine": cosine_similarity[off_diagonal].mean().item(),
    "mean_horizontal_neighbor_cosine": right_similarity.mean().item(),
    "mean_vertical_neighbor_cosine": down_similarity.mean().item(),
    "mean_nearest_token_cosine": nearest_similarity.mean().item(),
})

## How the decoder reads visual tokens

Generate a sequence, then run that sequence once without a cache to collect cross-attention weights. The resulting tensor has dimensions: decoder layers × heads × output positions × visual tokens.

In [ ]:
with torch.inference_mode():
    generated_ids = model.generate(  # ty: ignore[invalid-argument-type, missing-argument]
        encoder_outputs=encoder_outputs,
        decoder_input_ids=decoder_start_ids(model),
        max_new_tokens=MAX_NEW_TOKENS,
    )
    decoder_outputs = model(
        encoder_outputs=encoder_outputs,
        decoder_input_ids=generated_ids[:, :-1],
        output_attentions=True,
        use_cache=False,
        return_dict=True,
    )

decoded_tokens = donut.processor.tokenizer.convert_ids_to_tokens(generated_ids[0, 1:])
cross_attention = torch.stack([
    attention[0].float().cpu() for attention in decoder_outputs.cross_attentions
])
cross_attention.shape, decoded_tokens

In [ ]:
mean_attention = cross_attention.mean(dim=(0, 1))
positions = torch.linspace(0, len(decoded_tokens) - 1, steps=min(6, len(decoded_tokens))).long().unique()

fig, axes = plt.subplots(1, len(positions), figsize=(3 * len(positions), 3))
axes = [axes] if len(positions) == 1 else axes
for axis, position in zip(axes, positions.tolist()):
    attention_map = mean_attention[position].reshape(grid_height, grid_width)
    axis.imshow(attention_map, cmap="magma")
    axis.set_title(f"{position}: {decoded_tokens[position]}")
    axis.axis("off")
plt.tight_layout()

## Attention concentration

Normalized entropy near 1 means diffuse attention; lower values mean concentration on a small token subset. Effective tokens is `exp(entropy)`.

In [ ]:
probabilities = mean_attention.clamp_min(1e-12)
entropy = -(probabilities * probabilities.log()).sum(dim=1)
normalized_entropy = entropy / torch.tensor(tokens.shape[0]).log()
top_10_mass = probabilities.topk(min(10, tokens.shape[0]), dim=1).values.sum(dim=1)

pd.DataFrame({
    "position": range(len(decoded_tokens)),
    "token": decoded_tokens,
    "normalized_attention_entropy": normalized_entropy.numpy(),
    "effective_visual_tokens": entropy.exp().numpy(),
    "top_10_attention_mass": top_10_mass.numpy(),
})

In [ ]:
attention_received = cross_attention.mean(dim=(0, 1, 2))
norm_attention_correlation = torch.corrcoef(torch.stack([token_norms, attention_received]))[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(attention_received.reshape(grid_height, grid_width), cmap="magma")
axes[0].set_title("Mean attention received")
axes[1].scatter(token_norms, attention_received, s=8, alpha=0.5)
axes[1].set(title=f"Norm vs attention (r={norm_attention_correlation:.3f})", xlabel="Token norm", ylabel="Mean attention")
plt.tight_layout()